# Sentiment Analysis Using Bert Model Demonstration In Python

## Background

Text data such as movie reviews is unstructured and cannot be directly used by machine learning models. With advancements in Natural Language Processing, transformer-based models like BERT have enabled machines to understand context and sentiment effectively. The IMDB dataset is widely used for sentiment analysis, containing labeled movie reviews as positive or negative.

**BERT (Bidirectional Encoder Representations from Transformers)** is a transformer-based model that understands text context by processing words in both directions (left and right).
It is pre-trained on large text corpora and can be fine-tuned for tasks like sentiment analysis, classification, and question answering.

## Objective

To use a pre-trained BERT model to predict sentiment (positive/negative) on IMDB test data and analyze misclassifications by comparing predicted labels with actual labels.

## Dataset Description

The IMDB dataset is a benchmark dataset used for sentiment analysis tasks. It consists of 50,000 movie reviews, which are equally divided into training and test sets (25,000 each). Each review is labeled based on its sentiment as either positive or negative.

Features

text:
- The movie review written in natural language (unstructured text data).

label:

- Sentiment of the review:

> 0 → Negative

> 1 → Positive

### Import Libraries

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

### Load IMBD Dataset

**Note** 

*Training data is not used because a pre-trained and fine-tuned model is directly applied to test data for prediction and evaluation.*


In [ ]:
# Load dataset
dataset = load_dataset("imdb")

# Select test split
test_data = dataset["test"]

# taking sample 2000
test_sample = test_data.shuffle(seed=42).select(range(2000))

# Reduce size (optional)
df_test = test_sample.to_pandas()



In [6]:
df_test.head()

,text,label,bert_result,predicted,bert_score
0,<br /><br />When I unsuspectedly rented A Thou...,1,"{'label': 'POSITIVE', 'score': 0.9975194334983...",1,0.997519
1,This is the latest entry in the long series of...,1,"{'label': 'POSITIVE', 'score': 0.9980888962745...",1,0.998089
2,This movie was so frustrating. Everything seem...,0,"{'label': 'NEGATIVE', 'score': 0.9853643774986...",0,0.985364
3,"I was truly and wonderfully surprised at ""O' B...",1,"{'label': 'POSITIVE', 'score': 0.9979335069656...",1,0.997934
4,This movie spends most of its time preaching t...,0,"{'label': 'NEGATIVE', 'score': 0.9985034465789...",0,0.998503


In [3]:
df_test['label'].value_counts()

label
1    1000
0    1000
Name: count, dtype: int64

### Load Pre-trained Sentiment BERT 


In [4]:
from transformers import pipeline

bert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 1098.05it/s]


### Get Prediction

In [5]:
# Apply model on text column
df_test['bert_result'] = df_test['text'].apply(lambda x: bert_model(str(x)[:512])[0])

# Extract sentiment label
df_test['predicted'] = df_test['bert_result'].apply(lambda x: 1 if x['label'] == 'POSITIVE' else 0)

# Extract confidence score
df_test['bert_score'] = df_test['bert_result'].apply(lambda x: x['score'])

df_test[['text', 'label', 'predicted', 'bert_score']].sample(5)

,text,label,predicted,bert_score
236,"""Eighteen"" (2004) tells the story of Pip Ander...",1,0,0.915413
1818,"A Brazilian cable television is presenting ""Ya...",0,0,0.999713
299,Daft potboilers don't come much dafter than th...,0,1,0.997479
396,I recently saw this film at a 3-D film festiva...,1,1,0.999421
1762,A great suspenseful thriller the acting is fir...,1,1,0.999867


### Evaluation Metric : Classification Report

As actual labels are available in the test data, it is feasible to compute evaluation metrics like the classification report to assess model performance

In [6]:
from sklearn.metrics import classification_report

print(classification_report(df_test["label"], df_test["predicted"]))

              precision    recall  f1-score   support

           0       0.83      0.83      0.83       512
           1       0.82      0.82      0.82       488

    accuracy                           0.82      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.82      0.82      0.82      1000



In [7]:
misclassification_rate = (df_test["label"] != df_test["predicted"]).mean()
print("Misclassification Rate:", misclassification_rate)

Misclassification Rate: 0.175
